In [1]:
import os
import json
from collections import Counter
from datetime import datetime, timezone, timedelta
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
from tqdm.notebook import tqdm

### Configuration

In [15]:
ID = "038"
SEED = 42
level = "l1"
FEATURE_DIR = Path(f"../../artifacts/features/{ID}")

os.makedirs(FEATURE_DIR, exist_ok=True)

# === Chank処理の個数と処理する番目を確定 ===
CHUNK_SIZE = 5
CHUNK_N = 2

pd.set_option("display.max_rows", 500)
pd.set_option("display.max_columns", 500)
pl.Config.set_tbl_rows(500)
pl.Config.set_tbl_cols(500)

print(f"Feature dir created successfully in \n{FEATURE_DIR}")

Feature dir created successfully in 
../../artifacts/features/038


### Utils

In [3]:
def check_info(
    train: pl.DataFrame,
    test: pl.DataFrame
) -> tuple[float, float, float]:
    train_mem = sum(train[col].to_numpy().nbytes for col in train.columns) / 1024**3
    test_mem = sum(test[col].to_numpy().nbytes for col in test.columns) / 1024**3

    print("=== Shape & Memory ===")
    print(f"Train Shape: {train.shape}, Test Shape: {test.shape}")
    print(f"Train Memory: {train_mem:.2f} GB, Test Memory: {test_mem:.2f} GB\n")

    dtype_counts = Counter([str(dt) for dt in train.dtypes])

    n_cat = None
    print("=== DTypes ===")
    for dtype, cnt in dtype_counts.items():
        print(f"{dtype}: {cnt}")
        if dtype == "Categorical":
            n_cat = cnt
    return train_mem, test_mem, n_cat


def downcast(df: pl.DataFrame) -> pl.DataFrame:
    INT32_MIN, INT32_MAX = -2_147_483_648, 2_147_483_647

    df = df.with_columns(pl.col(pl.Float64).cast(pl.Float32))

    # Int64で安全に落とせる列だけ選別
    int64_cols = [c for c, dt in df.schema.items() if dt == pl.Int64]
    safe_cols = []
    for c in int64_cols:
        mn, mx = df[c].min(), df[c].max()
        if mn >= INT32_MIN and mx <= INT32_MAX:
            safe_cols.append(c)

    # 安全な列だけ Int32 に
    if safe_cols:
        df = df.with_columns(pl.col(safe_cols).cast(pl.Int32))
    return df

def add_bin_columns(
    df: pl.DataFrame,
    cols: list[str],
    n_bins: int,
    *,
    strategy: str = "quantile",
    quantile_method: str = "nearest",
    suffix: str = "_bin",
    drop_source: bool = False,
) -> pl.DataFrame:
    edges: dict[str, np.ndarray] = {}

    for c in cols:
        vals = df[c].to_numpy()
        vals = vals[np.isfinite(vals)]
        if vals.size == 0:
            continue

        if strategy == "quantile":
            qs = np.linspace(0.0, 1.0, n_bins + 1)
            qv = np.quantile(vals, qs, method=quantile_method)
            e = np.unique(qv)
        else:
            vmin = float(np.nanmin(vals))
            vmax = float(np.nanmax(vals))
            e = np.linspace(vmin, vmax, n_bins + 1)

        if e.size < 3:
            vmin = np.nanmin(vals)
            vmax = np.nanmax(vals)
            if not np.isfinite(vmin):
                vmin = 0.0
            if not np.isfinite(vmax):
                vmax = vmin + 1.0
            e = np.array([vmin - 1, vmax + 1], dtype="float64")

        edges[c] = e.astype("float64")

    out = df
    for c, e in edges.items():
        bname = f"{c}{suffix}"
        out = out.with_columns(
            pl.col(c)
              .cut(list(e))
              .cast(pl.Categorical)
              .to_physical()
              .rank("dense")
              .alias(bname)
        )
        if drop_source:
            out = out.drop(c)
    return out


def compute_group_stats(
    df: pl.DataFrame,
    key_cols: list[str],
    value_cols: list[str],
    *,
    stats: tuple[str, ...] = ("mean", "std", "min", "max", "median")
    ,
) -> pl.DataFrame:
    stats_dict = {}
    for k in tqdm(key_cols):
        for v in value_cols:
            if v in k:
                continue
            aggs = []
            col_names = []
            if "mean" in stats:
                aggs.append(pl.col(v).mean().alias(f"{v}_mean_by_{k}"))
                col_names.append(f"{v}_mean_by_{k}")
            if "std" in stats:
                aggs.append(pl.col(v).std(ddof=1).alias(f"{v}_std_by_{k}"))
                col_names.append(f"{v}_std_by_{k}")
            if "min" in stats:
                aggs.append(pl.col(v).min().alias(f"{v}_min_by_{k}"))
                col_names.append(f"{v}_min_by_{k}")
            if "max" in stats:
                aggs.append(pl.col(v).max().alias(f"{v}_max_by_{k}"))
                col_names.append(f"{v}_max_by_{k}")
            if "median" in stats:
                aggs.append(pl.col(v).median().alias(f"{v}_median_by_{k}"))
                col_names.append(f"{v}_median_by_{k}")

            grouped_df = (
                df.select([k, v])
                .group_by(k)
                .agg(aggs)
            )
            stats_array = (
                df.join(
                    grouped_df.select(col_names + [k]),
                    on=k,
                    how="left"
                )
                .select(col_names)
                .to_numpy()
                .astype(dtype=np.float32, copy=False)
            )
            for i, c in enumerate(col_names):
                stats_dict[c] = stats_array[:, i]

            del grouped_df, stats_array

    stats_df = pl.DataFrame(stats_dict)

    return stats_df

### Feature Engineering
- 037派生
- original dataを使って統計量を新たに計算

In [4]:
# === Load Data ===
train = pl.read_csv("../../input/train.csv").drop("id")
test = pl.read_csv("../../input/test.csv").drop("id")
orig = pl.read_parquet("../../input/original.parquet")
orig = orig.with_columns(
    pl.when(pl.col("y") == "yes").then(1)
      .when(pl.col("y") == "no").then(0)
      .otherwise(None)
      .alias("y")
)

y_tr = train["y"].cast(pl.Int8)
y_orig = orig["y"].cast(pl.Int8)
y_merged = pl.concat([y_tr, y_orig], how="vertical")

train = train.drop("y")
orig = orig.drop("y")

CATS = [col for col in train.columns if train[col].dtype == pl.Utf8]
NUMS = [col for col in train.columns if train[col].dtype != pl.Utf8]
print(f"NUMS: {len(NUMS)}\n{NUMS}")
print(f"\nCATS: {len(CATS)}\n{CATS}")

NUMS: 7
['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']

CATS: 9
['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']


In [5]:
# === 全データを結合 ===
all_data = pl.concat([train, test], how="vertical")
cat_exprs = [
    pl.col(c)
    .cast(pl.Categorical)
    .to_physical()
    .rank("dense")
    .cast(pl.Int32).alias(c)
    for c in CATS
]
all_data = all_data.with_columns(cat_exprs)
num_df = all_data.select(NUMS)
cat_df = all_data.select(CATS)

In [6]:
NUMS2CATS = [f"{c}2" for c in NUMS]
num2cat_exprs = [
    pl.col(c)
    .cast(pl.Utf8)
    .cast(pl.Categorical)
    .to_physical()
    .rank("dense")
    .cast(pl.Int32).alias(f"{c}2")
    for c in NUMS
]
num_df2 = num_df.select(num2cat_exprs)
print(f"num_df2: {len(num_df2.columns)}\n{num_df2.columns}")

orig_df = orig.with_columns(cat_exprs + num2cat_exprs)

num_df2: 7
['age2', 'balance2', 'day2', 'duration2', 'campaign2', 'pdays2', 'previous2']


In [7]:
# Grouped Dfを作成
stats_df = compute_group_stats(
    pl.concat([cat_df, num_df2, num_df], how="horizontal"),
    CATS + NUMS2CATS,
    NUMS,
    stats=("mean", "std", "max", "median")
)
print(f"Created {len(stats_df.columns)} new columns")

  0%|          | 0/16 [00:00<?, ?it/s]

Created 416 new columns


In [8]:
# original data用の関数
def compute_group_stats_orig(
    df: pl.DataFrame,
    value_df: pl.DataFrame,
    key_cols: list[str],
    value_cols: list[str],
    *,
    stats: tuple[str, ...] = ("mean", "std", "min", "max", "median")
    ,
) -> pl.DataFrame:
    stats_dict = {}
    for k in tqdm(key_cols):
        for v in value_cols:
            if v in k:
                continue
            aggs = []
            col_names = []
            if "mean" in stats:
                aggs.append(pl.col(v).mean().alias(f"{v}_mean_by_{k}2"))
                col_names.append(f"{v}_mean_by_{k}2")
            if "std" in stats:
                aggs.append(pl.col(v).std(ddof=1).alias(f"{v}_std_by_{k}2"))
                col_names.append(f"{v}_std_by_{k}2")
            if "min" in stats:
                aggs.append(pl.col(v).min().alias(f"{v}_min_by_{k}2"))
                col_names.append(f"{v}_min_by_{k}2")
            if "max" in stats:
                aggs.append(pl.col(v).max().alias(f"{v}_max_by_{k}2"))
                col_names.append(f"{v}_max_by_{k}2")
            if "median" in stats:
                aggs.append(pl.col(v).median().alias(f"{v}_median_by_{k}2"))
                col_names.append(f"{v}_median_by_{k}2")

            grouped_df = (
                value_df.select([k, v])
                .group_by(k)
                .agg(aggs)
            )
            stats_array = (
                df.join(
                    grouped_df.select(col_names + [k]),
                    on=k,
                    how="left"
                )
                .select(col_names)
                .to_numpy()
                .astype(dtype=np.float32, copy=False)
            )
            for i, c in enumerate(col_names):
                stats_dict[c] = stats_array[:, i]

            del grouped_df, stats_array

    stats_df = pl.DataFrame(stats_dict)

    return stats_df

In [9]:
# Original dataで統計量を算出
stats_df2 = compute_group_stats_orig(
    pl.concat([cat_df, num_df2, num_df], how="horizontal"),
    orig_df,
    CATS + NUMS2CATS,
    NUMS,
    stats=("mean", "std", "max", "median")
)
print(f"Created {len(stats_df.columns)} new columns")

  0%|          | 0/16 [00:00<?, ?it/s]

Created 416 new columns


In [10]:
# Durationのそれぞれの桁の数
duration_df = num_df.select(pl.col("duration"))

exprs = []
for k in (0, 1, 2, 3):
    exprs.append(
        ((pl.col("duration") // (10**k)) % 10)
        .cast(pl.Int32)
        .alias(f"duration_digitL{k}")
    )

duration_df = duration_df.with_columns(exprs).drop("duration")

In [11]:
# Merge Data
all_data = pl.concat([num_df, cat_df, stats_df, stats_df2, duration_df], how="horizontal")

In [12]:
# === row_id を追加 ===
all_data = all_data.with_row_index("row_id")

# === Downcast ===
all_data = downcast(all_data)

# === データを分割 ===
tr_df = all_data[:len(train)]
test_df = all_data[len(train):len(train)+len(test)]

# === targetを追加 ===
tr_df = tr_df.with_columns(y_tr.alias("target"))

### Add Fold Col

In [13]:
# Add Fold Col
folds_path = "../../artifacts/folds/folds.parquet"
pairs = [
    ("skf/k=5/s=42@train", "5fold-s42")
]
cfgs = [c for c, _ in pairs]
rename_map = {c: n for c, n in pairs}

# folds をまとめて読み → ワイド化（cfg列を列見出しに）→ 列名をfold_nameにリネーム
folds_wide = (
    pl.scan_parquet(folds_path)
      .filter(pl.col("cfg").is_in(cfgs))
      .unique(subset=["row_id", "cfg"], keep="last")
      .select(["row_id", "cfg", "fold"])
      .collect(engine="streaming")
      .pivot(values="fold", index="row_id", columns="cfg", aggregate_function="first")
      .rename(rename_map)
      .with_columns(pl.col("row_id").cast(pl.Int32))
      .with_columns([pl.all().exclude("row_id").cast(pl.Int8)])  # 型を軽く
)

# tr_df が DataFrame の場合
tr_df = tr_df.join(folds_wide, on="row_id", how="left")

/tmp/ipykernel_18385/4243328503.py:11: DeprecationWarning: the argument `columns` for `DataFrame.pivot` is deprecated. It was renamed to `on` in version 1.0.0.
  pl.scan_parquet(folds_path)


In [14]:
# === 特徴量エンジニアリング後の情報 ===
train_mem, test_mem, n_cat = check_info(tr_df, test_df)

=== Shape & Memory ===
Train Shape: (750000, 855), Test Shape: (250000, 853)
Train Memory: 2.38 GB, Test Memory: 0.79 GB

=== DTypes ===
UInt32: 1
Int32: 20
Float32: 832
Int8: 2


In [17]:
# === Save Data ===
tr_path = FEATURE_DIR / "train.parquet"
test_path = FEATURE_DIR / "test.parquet"

tr_df.write_parquet(tr_path)
test_df.write_parquet(test_path)

print(f"tr_df saved successfully to {tr_path}")
print(f"test_df saved successfully to {test_path}")

tr_df saved successfully to ../../artifacts/features/038/train.parquet
test_df saved successfully to ../../artifacts/features/038/test.parquet


### Save Meta data

In [16]:
JST = timezone(timedelta(hours=9))
meta = {
    "data_id": ID,
    "train_paths": [str(FEATURE_DIR / "train.parquet")],
    "test_paths": [str(FEATURE_DIR / "test.parquet")],
    "level": level,
    "created_at": datetime.now(JST).isoformat(),
    "train_shape": [tr_df.height, tr_df.width],
    "test_shape": [test_df.height, test_df.width],
    "memory": {
        "train": train_mem,
        "test": test_mem
    },
    "fold_column": pairs,
    "cat_cols": n_cat if n_cat else None
}

with open(f"{FEATURE_DIR}/meta.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

In [ ]:
# RAPIDS-based CV trainer (cuML)
from dataclasses import dataclass, field
from typing import Optional, List
import os, gc
import numpy as np
import polars as pl
import cupy as cp

from time import perf_counter as now
from sklearn.metrics import roc_auc_score, log_loss

from cuml.svm import SVC as cuSVC
from cuml.linear_model import LogisticRegression as cuLogReg
from cuml.ensemble import RandomForestClassifier as cuRF  # 使いたければ
# from cuml.preprocessing import StandardScaler  # 必要ならこちらでも可

# あなたのユーティリティ
# - compute_feature_stats(paths, features, num_cols, fold_col, exclude_folds)
# - CVLogger / NoOpLogger / CVResult / print_duration / free_ram_gib / free_vram_gib
# をそのまま利用します

@dataclass(eq=False)
class RAPIDSCVTrainer:
    data_id: str
    train_paths: str | list[str]
    test_paths: str | list[str]

    features: Optional[list[str]] = None
    target: str = "target"
    fold_col: Optional[str] = None
    weight_col: Optional[str] = None
    cat_cols: Optional[list[str]] = None  # 数値化済み前提

    # --- モデル選択とハイパラ ---
    params: dict = field(default_factory=dict)
    estimator: str = "svm_rbf"   # "svm_rbf" | "logreg" | "rf"

    n_fold: int = 5
    seed: int = 42

    # lazy scans
    lf_train: pl.LazyFrame = field(init=False)
    lf_test: pl.LazyFrame = field(init=False)

    def __post_init__(self):
        # normalize paths
        if isinstance(self.train_paths, (str, os.PathLike)):
            self.train_paths = [str(self.train_paths)]
        else:
            self.train_paths = [str(p) for p in self.train_paths]
        if isinstance(self.test_paths, (str, os.PathLike)):
            self.test_paths = [str(self.test_paths)]
        else:
            self.test_paths = [str(p) for p in self.test_paths]

        self.lf_train = pl.scan_parquet(self.train_paths)
        self.lf_test = pl.scan_parquet(self.test_paths)

        # 推奨デフォルト
        if self.estimator == "svm_rbf":
            default_params = dict(
                C=3.0,
                gamma="scale",     # 'scale' or 'auto' or float
                kernel="rbf",
                cache_size=1024,   # MB
                max_iter=-1,
                tol=1e-3,
                probability=False  # cuML SVC は proba 非対応のことが多い
            )
        elif self.estimator == "logreg":
            default_params = dict(
                C=1.0,
                penalty="l2",
                max_iter=2000,
                tol=1e-4,
                fit_intercept=True,
                l1_ratio=None,
            )
        elif self.estimator == "rf":
            default_params = dict(
                n_estimators=500,
                max_depth=16,
                n_streams=8,
                bootstrap=True
            )
        else:
            raise ValueError("estimator must be one of ['svm_rbf','logreg','rf']")

        self.params = {**default_params, **(self.params or {})}

        # 列メタ
        hdr = pl.read_parquet(self.train_paths, n_rows=0)
        all_cols = hdr.columns

        if self.fold_col is None:
            self.fold_col = f"{self.n_fold}fold-s{self.seed}"
        if self.fold_col not in all_cols:
            raise ValueError(f"fold_col not found: {self.fold_col}")

        if self.features is None:
            meta = {"row_id", self.fold_col}
            if self.target in all_cols: meta.add(self.target)
            if self.weight_col in all_cols: meta.add(self.weight_col)
            self.features = [c for c in all_cols if c not in meta and "fold" not in c]

        # 数値/カテゴリ（数値化済み）分割
        if self.cat_cols is None:
            # Categorical dtype を見つけても、学習前に必ず数値化しておいてね
            self.cat_cols = [c for c, dt in zip(hdr.columns, hdr.dtypes) if dt == pl.Categorical and c in self.features]

        self.num_cols = [c for c in self.features if c not in self.cat_cols]
        self.num_idxs = [self.features.index(c) for c in self.num_cols]

    # ------------------------------

    def _make_model(self, n_features: int):
        p = dict(self.params)
        if self.estimator == "svm_rbf":
            # gamma の 'scale' / 'auto' を数値に（標準化後を想定）
            g = p.get("gamma", "scale")
            if isinstance(g, str):
                if g == "scale":
                    p["gamma"] = 1.0 / max(1, n_features)  # var≈1 想定
                elif g == "auto":
                    p["gamma"] = 1.0 / max(1, n_features)
                else:
                    raise ValueError("gamma must be float/'scale'/'auto'")
            return cuSVC(**p)
        elif self.estimator == "logreg":
            return cuLogReg(**p)
        elif self.estimator == "rf":
            return cuRF(**p)

    # ------------------------------

    def fit(self, loggers: List = None):
        if self.test_paths is None:
            raise ValueError("Please provide test_paths.")

        t0 = now()
        loggers = loggers or []

        # 行数
        train_rows = self.lf_train.select(pl.len()).collect().item()
        test_rows = self.lf_test.select(pl.len()).collect().item()

        oof = np.zeros(train_rows, dtype=np.float32)
        test_pred = np.zeros(test_rows, dtype=np.float32)

        fold_aucs, fold_lls = [], []

        for fold in range(self.n_fold):
            print("=" * 22)
            print(f"===== Fold {fold+1} / {self.n_fold} =====")

            need_cols = self.features + [self.target, "row_id"]

            train_df = (
                self.lf_train
                .filter(pl.col(self.fold_col) != fold)
                .select(need_cols)
                .collect(engine="streaming")
            )
            valid_df = (
                self.lf_train
                .filter(pl.col(self.fold_col) == fold)
                .select(need_cols)
                .collect(engine="streaming")
            )
            test_df = (
                self.lf_test
                .select(self.features)
                .collect(engine="streaming")
            )

            # numpy → GPU
            Xtr = train_df.select(self.features).to_numpy().astype(np.float32, copy=False)
            ytr = train_df.select(self.target).to_numpy().ravel().astype(np.int32, copy=False)
            Xva = valid_df.select(self.features).to_numpy().astype(np.float32, copy=False)
            yva = valid_df.select(self.target).to_numpy().ravel().astype(np.int32, copy=False)
            rid_va = valid_df.select("row_id").to_numpy().ravel().astype(np.int32, copy=False)

            Xte = test_df.select(self.features).to_numpy().astype(np.float32, copy=False)

            # 標準化（train 統計）
            mean, std = compute_feature_stats(
                self.train_paths, self.features, self.num_cols, self.fold_col, exclude_folds=[fold]
            )
            # CPU -> GPU 配列
            Xtr_g = cp.asarray(Xtr)
            Xva_g = cp.asarray(Xva)
            Xte_g = cp.asarray(Xte)

            if len(self.num_idxs) > 0:
                mu_g = cp.asarray(mean); sd_g = cp.asarray(std)
                Xtr_g[:, self.num_idxs] = (Xtr_g[:, self.num_idxs] - mu_g) / sd_g
                Xva_g[:, self.num_idxs]  = (Xva_g[:, self.num_idxs]  - mu_g) / sd_g
                Xte_g[:, self.num_idxs]  = (Xte_g[:, self.num_idxs]  - mu_g) / sd_g

            # モデル
            model = self._make_model(n_features=Xtr_g.shape[1])

            # 学習
            model.fit(Xtr_g, cp.asarray(ytr))

            # 予測スコア（確率 or margin）
            def infer_score(m, Xg):
                if hasattr(m, "predict_proba"):
                    proba = m.predict_proba(Xg)
                    # cuML は cupy を返すので CPU に戻す
                    return cp.asnumpy(proba)[:, 1].astype(np.float32, copy=False)
                elif hasattr(m, "decision_function"):
                    s = m.decision_function(Xg)
                    return cp.asnumpy(s).astype(np.float32, copy=False)
                else:
                    # 最終手段：predict（0/1）。AUCには向かない
                    s = m.predict(Xg)
                    return cp.asnumpy(s).astype(np.float32, copy=False)

            va_score = infer_score(model, Xva_g)
            te_score = infer_score(model, Xte_g)

            # OOF
            oof[rid_va] = va_score
            # テストは平均
            test_pred += te_score / self.n_fold

            # metric（AUC）
            auc = roc_auc_score(yva, va_score)
            fold_aucs.append(auc)
            print(f"Fold {fold} AUC: {auc:.5f}")

            # logloss（確率でない場合は未算出）
            if self.estimator == "logreg" and va_score.min() >= 0 and va_score.max() <= 1:
                ll = log_loss(yva, va_score, labels=[0,1])
                fold_lls.append(ll)
                print(f"Fold {fold} LogLoss: {ll:.5f}")

            # GC
            del train_df, valid_df, test_df, Xtr, Xva, Xte, Xtr_g, Xva_g, Xte_g
            gc.collect()
            cp.get_default_memory_pool().free_all_blocks()

        # OOF 評価
        y_all = (
            pl.read_parquet(self.train_paths, columns=[self.target])
            .get_column(self.target).cast(pl.Int32).to_numpy()
        )
        oof_auc = roc_auc_score(y_all, oof)
        print(f"\n=== CV Results ===\nOOF AUC: {oof_auc:.5f}")
        if fold_lls:
            oof_ll = log_loss(y_all, oof) if (oof.min()>=0 and oof.max()<=1) else float("nan")
            print(f"OOF LogLoss: {oof_ll:.5f}")

        res = CVResult(oof=oof, test_pred=test_pred, score=oof_auc)
        return res
